In [24]:
import pandas as pd

# Load the dataset
batting_cleaned = pd.read_csv('/content/ganjoo_ipl_mbm_bat_final.csv')

# Convert column names to lowercase for consistency
batting_cleaned.columns = batting_cleaned.columns.str.lower()

# Display actual column names
#print("Columns in dataset:", batting_cleaned.columns)

# Define required columns
required_columns = {'date', 'match_id', 'bat_team', 'bowl_team', 'inns', 'batter_id',
                    'batter', 'runs', 'balls', 'sr', 'isout', 'wickettype', 'bowler_id',
                    'bowler', 'venue', 'boundaries', 'fours', 'sixes'}

# Fill missing columns with default values
for col in required_columns:
    if col not in batting_cleaned.columns:
        if col == 'isout':
            batting_cleaned[col] = False  # Default: not out (boolean)
        elif col == 'wickettype':
            batting_cleaned[col] = "Unknown"
        elif col == 'boundaries':
            batting_cleaned[col] = batting_cleaned.get('fours', 0) + batting_cleaned.get('sixes', 0)

# Convert 'date' column to datetime format
batting_cleaned['date'] = pd.to_datetime(batting_cleaned['date'])

# Convert 'isOut' from string/boolean to integer (1 for True, 0 for False)
batting_cleaned['isout'] = batting_cleaned['isout'].astype(str).map({'True': 1, 'False': 0}).fillna(0).astype(int)

# Compute boundaries
batting_cleaned['boundaries'] = batting_cleaned['fours'] + batting_cleaned['sixes']

# Sort data by batter and match date
batting_cleaned.sort_values(by=['batter', 'date'], inplace=True)

# Ensure numeric types
batting_cleaned['runs'] = pd.to_numeric(batting_cleaned['runs'], errors='coerce').fillna(0)
batting_cleaned['balls'] = pd.to_numeric(batting_cleaned['balls'], errors='coerce').fillna(0)

# Compute cumulative stats for each batter
batting_cleaned['cumulative_runs'] = batting_cleaned.groupby('batter')['runs'].cumsum().shift()
batting_cleaned['cumulative_dismissals'] = batting_cleaned.groupby('batter')['isout'].cumsum().shift()
batting_cleaned['cumulative_matches'] = batting_cleaned.groupby('batter')['batter'].cumcount()
batting_cleaned['cumulative_balls'] = batting_cleaned.groupby('batter')['balls'].cumsum().shift()
batting_cleaned['cumulative_boundaries'] = batting_cleaned.groupby('batter')['boundaries'].cumsum().shift()

# Compute batting statistics
batting_cleaned['batter_avg'] = batting_cleaned['cumulative_runs'] / batting_cleaned['cumulative_dismissals'].replace(0, pd.NA)
batting_cleaned['batter_rpm'] = batting_cleaned['cumulative_runs'] / batting_cleaned['cumulative_matches'].replace(0, pd.NA)
batting_cleaned['batter_sr'] = (batting_cleaned['cumulative_runs'] * 100) / batting_cleaned['cumulative_balls'].replace(0, pd.NA)
batting_cleaned['batter_bpm'] = batting_cleaned['cumulative_boundaries'] / batting_cleaned['cumulative_matches'].replace(0, pd.NA)

# Ensure first match has NaN (null) values
for col in ['batter_avg', 'batter_rpm', 'batter_sr', 'batter_bpm']:
    batting_cleaned.loc[batting_cleaned.groupby('batter').head(1).index, col] = pd.NA

# Drop unnecessary columns
batting_cleaned.drop(columns=['cumulative_runs', 'cumulative_dismissals', 'cumulative_matches',
                              'cumulative_boundaries', 'cumulative_balls'], inplace=True)
# Ensure first match has NaN (null) values
batting_cleaned.loc[
    batting_cleaned.groupby('batter').head(1).index, 'batter_avg'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby('batter').head(1).index, 'batter_rpm'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby('batter').head(1).index, 'batter_sr'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby('batter').head(1).index, 'batter_bpm'
] = pd.NA



# Print final dataset columns
print("Final dataset columns:", batting_cleaned.head())
batting_cleaned.to_csv('/content/batting_cleaned2.csv', index=False)



Final dataset columns:           date  match_id                     bat_team              bowl_team  \
100 2015-04-11    829713  Royal Challengers Bangalore  Kolkata Knight Riders   
156 2015-04-13    829719  Royal Challengers Bangalore    Sunrisers Hyderabad   
365 2015-04-19    829737  Royal Challengers Bangalore         Mumbai Indians   
458 2015-04-22    829745  Royal Challengers Bangalore    Chennai Super Kings   
495 2015-04-24    829749  Royal Challengers Bangalore       Rajasthan Royals   

     inns  batter_id          batter  runs  balls  fours  ...  isout  \
100     2      44936  AB de Villiers    28     13      3  ...      1   
156     1      44936  AB de Villiers    46     28      5  ...      1   
365     2      44936  AB de Villiers    41     11      5  ...      1   
458     2      44936  AB de Villiers    14     10      0  ...      0   
495     2      44936  AB de Villiers    47     34      6  ...      0   

     wickettype       bowler_id          bowler                

In [26]:
# Columns to apply imputation
cols = ['batter_avg', 'batter_sr', 'batter_rpm', 'batter_bpm']

# Loop through columns and apply transformations
for col in cols:
    # Backward fill within each batsman's group
    batting_cleaned[col] = batting_cleaned.groupby('batter')[col].transform(lambda x: x.bfill())

    # Forward fill within each batsman's group
    batting_cleaned[col] = batting_cleaned.groupby('batter')[col].transform(lambda x: x.ffill())

    # Calculate the median of the entire column (across all batsmen)
    median = batting_cleaned[col].median()

    # Fill any remaining NaNs within each batsman's group with a lower median value
    batting_cleaned[col] = batting_cleaned.groupby('batter')[col].fillna(median / 2)

<ipython-input-26-14c9a33123fd>:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  batting_cleaned[col] = batting_cleaned.groupby('batter')[col].transform(lambda x: x.bfill())
<ipython-input-26-14c9a33123fd>:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  batting_cleaned[col] = batting_cleaned.groupby('batter')[col].transform(lambda x: x.ffill())
<ipython-input-26-14c9a33123fd>:16: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want 

In [59]:
# Sort batting_cleaned by bat_team and match date
batting_cleaned.sort_values(by=['bat_team', 'date'], inplace=True)

# Compute total runs scored by team per match
team_runs_per_match = batting_cleaned.groupby(['match_id', 'bat_team'])['runs'].sum().reset_index()

# Extract match date directly from batting_cleaned
team_runs_per_match = team_runs_per_match.merge(
    batting_cleaned[['match_id', 'date']].drop_duplicates(),
    on='match_id',
    how='left'
)

# Sort team_runs_per_match by team and match date
team_runs_per_match.sort_values(by=['bat_team', 'date'], inplace=True)

# Compute cumulative runs for the team before each match
team_runs_per_match['cumulative_team_runs'] = (
    team_runs_per_match.groupby('bat_team')['runs']
    .cumsum().shift()
)

# Compute the number of past matches played by the team
team_runs_per_match['cumulative_team_matches'] = (
    team_runs_per_match.groupby('bat_team').cumcount()
)

# Compute batting team average runs before the match
team_runs_per_match['bat_team_avg'] = (
    team_runs_per_match['cumulative_team_runs'] / team_runs_per_match['cumulative_team_matches'].replace(0, pd.NA)
)

# Ensure first match for each team is NaN (null)
team_runs_per_match.loc[
    team_runs_per_match.groupby('bat_team').head(1).index, 'bat_team_avg'
] = pd.NA

# ✅ Drop existing 'bat_team_avg' before merging to prevent duplicate columns
batting_cleaned.drop(columns=['bat_team_avg'], errors='ignore', inplace=True)

# Merge the computed values back into batting_cleaned
batting_cleaned = batting_cleaned.merge(
    team_runs_per_match[['match_id', 'bat_team', 'bat_team_avg']],
    on=['match_id', 'bat_team'],
    how='left'
)

# Save the cleaned data
batting_cleaned.to_csv('/content/batting_cleanedFix.csv', index=False)

print("Merging successful! Check 'batting_cleanedFix.csv'.")


Merging successful! Check 'batting_cleanedFix.csv'.


In [32]:
batting_cleaned['bat_team_avg'] = batting_cleaned.groupby('bat_team')['bat_team_avg'].transform(lambda x: x.bfill())


<ipython-input-32-ce8cbf3bc16a>:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  batting_cleaned['bat_team_avg'] = batting_cleaned.groupby('bat_team')['bat_team_avg'].transform(lambda x: x.bfill())


In [61]:
batting_cleaned = pd.read_csv('/content/batting_cleaned_teamStats.csv')
print(batting_cleaned.head())



         date  match_id             bat_team       bowl_team  inns  batter_id  \
0  2015-04-09    829707  Chennai Super Kings  Delhi Capitals     1      31820   
1  2015-04-09    829707  Chennai Super Kings  Delhi Capitals     1      37737   
2  2015-04-09    829707  Chennai Super Kings  Delhi Capitals     1      51439   
3  2015-04-09    829707  Chennai Super Kings  Delhi Capitals     1      53118   
4  2015-04-09    829707  Chennai Super Kings  Delhi Capitals     1      44828   

             batter  runs  balls  fours  ...               bowler  \
0      Ashish Nehra     0      0      0  ...                  NaN   
1  Brendon McCullum     4      2      1  ...  Nathan Coulter-Nile   
2      Dwayne Bravo     1      3      0  ...    Domnic Muthuswami   
3      Dwayne Smith    34     31      6  ...          Imran Tahir   
4    Faf du Plessis    32     24      3  ...     Jean-Paul Duminy   

                    venue  boundaries batter_avg batter_rpm   batter_sr  \
0  MA Chidambaram Stadi

In [66]:
batting_cleaned.drop(columns=['bat_team_avg_x', 'bat_team_avg_y'], errors='ignore', inplace=True)

# Display the first few rows to confirm the changes
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleanedTS.csv', index=False)



             date  match_id             bat_team              bowl_team  inns  \
14263  2024-05-26   1426312  Sunrisers Hyderabad  Kolkata Knight Riders     1   
14264  2024-05-26   1426312  Sunrisers Hyderabad  Kolkata Knight Riders     1   
14265  2024-05-26   1426312  Sunrisers Hyderabad  Kolkata Knight Riders     1   
14266  2024-05-26   1426312  Sunrisers Hyderabad  Kolkata Knight Riders     1   
14267  2024-05-26   1426312  Sunrisers Hyderabad  Kolkata Knight Riders     1   

       batter_id          batter  runs  balls  fours  ...  wickettype  \
14263     489889     Pat Cummins    24     19      2  ...      caught   
14264     446763  Rahul Tripathi     9     13      1  ...      caught   
14265    1159711   Shahbaz Ahmed     8      7      0  ...      caught   
14266     802575     T Natarajan     0      0      0  ...         DNB   
14267     530011     Travis Head     0      1      0  ...      caught   

                bowler_id              bowler                   venue  \
1

In [68]:
# Sort by batter, opposition, and date
batting_cleaned.sort_values(by=['batter', 'bowl_team', 'date'], inplace=True)

# Compute cumulative runs and dismissals before the match
batting_cleaned['cumulative_runs_vs_opp'] = (
    batting_cleaned.groupby(['batter', 'bowl_team'])['runs'].cumsum().shift(fill_value=0)
)

batting_cleaned['cumulative_dismissals_vs_opp'] = (
    batting_cleaned.groupby(['batter', 'bowl_team'])['isout'].cumsum().shift(fill_value=0)
)

batting_cleaned['cumulative_matches_vs_opp'] = (
    batting_cleaned.groupby(['batter', 'bowl_team'])['batter'].cumcount()
)

batting_cleaned['cumulative_boundaries_vs_opp'] = (
    batting_cleaned.groupby(['batter', 'bowl_team'])['boundaries'].cumsum().shift(fill_value=0)
)

batting_cleaned['cumulative_balls_vs_opp'] = (
    batting_cleaned.groupby(['batter', 'bowl_team'])['balls'].cumsum().shift(fill_value=0)
)

# Calculate batting average vs opposition before the match
batting_cleaned['batter_avg_opp'] = (
    batting_cleaned['cumulative_runs_vs_opp'] / batting_cleaned['cumulative_dismissals_vs_opp'].replace(0, pd.NA)
)

batting_cleaned['batter_rpm_opp'] = (
    batting_cleaned['cumulative_runs_vs_opp'] / batting_cleaned['cumulative_matches_vs_opp']
)

batting_cleaned['batter_bpm_opp'] = (
    batting_cleaned['cumulative_boundaries_vs_opp'] / batting_cleaned['cumulative_matches_vs_opp']
)

batting_cleaned['batter_sr_opp'] = (
    (batting_cleaned['cumulative_runs_vs_opp'] * 100) / batting_cleaned['cumulative_balls_vs_opp'].replace(0, pd.NA)
)

# Ensure first match vs opposition has NaN
batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'bowl_team']).head(1).index, 'batter_avg_opp'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'bowl_team']).head(1).index, 'batter_rpm_opp'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'bowl_team']).head(1).index, 'batter_bpm_opp'
] = pd.NA

batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'bowl_team']).head(1).index, 'batter_sr_opp'
] = pd.NA

# Drop unnecessary columns
batting_cleaned.drop(columns=['cumulative_balls_vs_opp', 'cumulative_runs_vs_opp',
                              'cumulative_dismissals_vs_opp', 'cumulative_matches_vs_opp',
                              'cumulative_boundaries_vs_opp'], inplace=True)

# Save the cleaned data
batting_cleaned.to_csv('/content/batting_cleaned_updated.csv', index=False)


In [69]:
# Columns to apply imputation
cols = ['batter_avg_opp', 'batter_sr_opp', 'batter_rpm_opp', 'batter_bpm_opp']

# Apply imputation efficiently
for col in cols:
    # Perform both backward and forward fill using transform (keeps original index)
    batting_cleaned[col] = (
        batting_cleaned.groupby(['batter', 'bowl_team'])[col]
        .transform(lambda x: x.bfill().ffill())
    )
    batting_cleaned[col] = (
        batting_cleaned.groupby(['batter'])[col]
        .transform(lambda x: x.bfill().ffill())
    )
    batting_cleaned[col] = (
        batting_cleaned.groupby(['bowl_team'])[col]
        .transform(lambda x: x.bfill().ffill())
    )

    # Compute median once
    median = batting_cleaned[col].median()

    # Fill remaining NaNs with median/2
    batting_cleaned[col].fillna(median / 2, inplace=True)


<ipython-input-69-92dc60c01e8d>:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.bfill().ffill())
<ipython-input-69-92dc60c01e8d>:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.bfill().ffill())
<ipython-input-69-92dc60c01e8d>:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lamb

In [73]:
# Sort by team, opposition, and match date
batting_cleaned.sort_values(by=['bat_team', 'bowl_team', 'date'], inplace=True)

# Compute total runs scored by team per match against opposition
team_vs_opp_runs = batting_cleaned.groupby(['match_id', 'bat_team', 'bowl_team'])['runs'].sum().reset_index()

# Extract match date directly from batting_cleaned
team_vs_opp_runs = team_vs_opp_runs.merge(
    batting_cleaned[['match_id', 'date']].drop_duplicates(),
    on='match_id',
    how='left'
)

# Sort by team, opposition, and match date
team_vs_opp_runs.sort_values(by=['bat_team', 'bowl_team', 'date'], inplace=True)

# Compute cumulative runs against opposition before match
team_vs_opp_runs['cumulative_team_runs_vs_opp'] = (
    team_vs_opp_runs.groupby(['bat_team', 'bowl_team'])['runs'].cumsum().shift()
)

# Compute the number of past matches vs opposition
team_vs_opp_runs['cumulative_team_matches_vs_opp'] = (
    team_vs_opp_runs.groupby(['bat_team', 'bowl_team']).cumcount()
)

# Calculate batting team average vs opposition before match
team_vs_opp_runs['bat_team_avg_opp'] = (
    team_vs_opp_runs['cumulative_team_runs_vs_opp'] /
    team_vs_opp_runs['cumulative_team_matches_vs_opp'].replace(0, pd.NA)
)

# Ensure first match vs opposition is NaN
team_vs_opp_runs.loc[
    team_vs_opp_runs.groupby(['bat_team', 'bowl_team']).head(1).index, 'bat_team_avg_opp'
] = pd.NA

# Merge back into batting_cleaned
batting_cleaned = batting_cleaned.merge(
    team_vs_opp_runs[['match_id', 'bat_team', 'bowl_team', 'bat_team_avg_opp']],
    on=['match_id', 'bat_team', 'bowl_team'],
    how='left'
)

print(batting_cleaned.tail())

batting_cleaned.to_csv('/content/batting_cleaned_updatedTSopp.csv', index=False)

             date  match_id             bat_team                    bowl_team  \
14263  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14264  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14265  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14266  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14267  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   

       inns  batter_id              batter  runs  balls  fours  ...  \
14263     2    1175496  Nitish Kumar Reddy    13     13      0  ...   
14264     2     489889         Pat Cummins    31     15      1  ...   
14265     2    1159711       Shahbaz Ahmed    40     37      1  ...   
14266     2     802575         T Natarajan     0      0      0  ...   
14267     2     530011         Travis Head     1      3      0  ...   

        batter_sr  batter_bpm  bat_team_avg batter_avg_opp batter_rpm_opp  \
14263  15

In [75]:
batting_cleaned.drop(columns=['bat_team_avg_opp_x', 'bat_team_avg_opp_y'], errors='ignore', inplace=True)

# Display the first few rows to confirm the changes
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleanedTSopp2.csv', index=False)

             date  match_id             bat_team                    bowl_team  \
14263  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14264  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14265  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14266  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14267  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   

       inns  batter_id              batter  runs  balls  fours  ...  \
14263     2    1175496  Nitish Kumar Reddy    13     13      0  ...   
14264     2     489889         Pat Cummins    31     15      1  ...   
14265     2    1159711       Shahbaz Ahmed    40     37      1  ...   
14266     2     802575         T Natarajan     0      0      0  ...   
14267     2     530011         Travis Head     1      3      0  ...   

       batter_avg  batter_rpm   batter_sr batter_bpm bat_team_avg  \
14263   57.500000

In [77]:
# Compute total runs scored by all teams per match against opposition
all_teams_vs_opp_runs = batting_cleaned.groupby(['match_id', 'bowl_team'])['runs'].sum().reset_index()

# Extract match date directly from batting_cleaned
all_teams_vs_opp_runs = all_teams_vs_opp_runs.merge(
    batting_cleaned[['match_id', 'date']].drop_duplicates(),
    on='match_id',
    how='left'
)

# Sort by opposition and match date
all_teams_vs_opp_runs.sort_values(by=['bowl_team', 'date'], inplace=True)

# Compute cumulative runs against opposition before match
all_teams_vs_opp_runs['cumulative_all_teams_runs_vs_opp'] = (
    all_teams_vs_opp_runs.groupby(['bowl_team'])['runs'].cumsum().shift()
)

# Compute the number of past matches vs opposition
all_teams_vs_opp_runs['cumulative_all_teams_matches_vs_opp'] = (
    all_teams_vs_opp_runs.groupby(['bowl_team']).cumcount()
)

# Calculate all teams' average runs vs opposition before match
all_teams_vs_opp_runs['all_team_avg_opp'] = (
    all_teams_vs_opp_runs['cumulative_all_teams_runs_vs_opp'] /
    all_teams_vs_opp_runs['cumulative_all_teams_matches_vs_opp'].replace(0, pd.NA)
)

# Ensure first match vs opposition is NaN
all_teams_vs_opp_runs.loc[
    all_teams_vs_opp_runs.groupby(['bowl_team']).head(1).index, 'all_team_avg_opp'
] = pd.NA

# Merge back into batting_cleaned
batting_cleaned = batting_cleaned.merge(
    all_teams_vs_opp_runs[['match_id', 'bowl_team', 'all_team_avg_opp']],
    on=['match_id', 'bowl_team'],
    how='left'
)
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleanedAllopp.csv', index=False)

             date  match_id             bat_team                    bowl_team  \
14263  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14264  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14265  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14266  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14267  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   

       inns  batter_id              batter  runs  balls  fours  ...  \
14263     2    1175496  Nitish Kumar Reddy    13     13      0  ...   
14264     2     489889         Pat Cummins    31     15      1  ...   
14265     2    1159711       Shahbaz Ahmed    40     37      1  ...   
14266     2     802575         T Natarajan     0      0      0  ...   
14267     2     530011         Travis Head     1      3      0  ...   

        batter_sr  batter_bpm  bat_team_avg batter_avg_opp batter_rpm_opp  \
14263  15

In [88]:
batting_cleaned=pd.read_csv('/content/batting_cleanedAllopp.csv')
batting_cleaned.drop(columns=['all_team_avg_opp_y'], errors='ignore', inplace=True)

# Display the first few rows to confirm the changes
print(batting_cleaned.tail())
#batting_cleaned.to_csv('/content/batting_cleanedTSopp2.csv', index=False)

# Apply backward fill and forward fill for 'bat_team_avg_opp' within groups
batting_cleaned['bat_team_avg_opp'] = batting_cleaned.groupby(['bat_team', 'bowl_team'])['bat_team_avg_opp'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg_opp'] = batting_cleaned.groupby(['bat_team', 'bowl_team'])['bat_team_avg_opp'].transform(lambda x: x.ffill())
batting_cleaned['bat_team_avg_opp'] = batting_cleaned.groupby(['bowl_team'])['bat_team_avg_opp'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg_opp'] = batting_cleaned.groupby(['bowl_team'])['bat_team_avg_opp'].transform(lambda x: x.ffill())
batting_cleaned['bat_team_avg_opp'] = batting_cleaned.groupby(['bat_team'])['bat_team_avg_opp'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg_opp'] = batting_cleaned.groupby(['bat_team'])['bat_team_avg_opp'].transform(lambda x: x.ffill())

# Apply backward fill and forward fill for 'all_team_avg_opp' within opposition groups
batting_cleaned['all_team_avg_opp_x'] = batting_cleaned.groupby(['bowl_team'])['all_team_avg_opp_x'].transform(lambda x: x.bfill())
batting_cleaned['all_team_avg_opp_x'] = batting_cleaned.groupby(['bowl_team'])['all_team_avg_opp_x'].transform(lambda x: x.ffill())
batting_cleaned.rename(columns={'all_team_avg_opp_x': 'all_team_avg_opp'}, inplace=True)
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleanedAllopp2.csv', index=False)


             date  match_id             bat_team                    bowl_team  \
14263  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14264  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14265  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14266  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   
14267  2024-04-25   1426279  Sunrisers Hyderabad  Royal Challengers Bengaluru   

       inns  batter_id              batter  runs  balls  fours  ...  \
14263     2    1175496  Nitish Kumar Reddy    13     13      0  ...   
14264     2     489889         Pat Cummins    31     15      1  ...   
14265     2    1159711       Shahbaz Ahmed    40     37      1  ...   
14266     2     802575         T Natarajan     0      0      0  ...   
14267     2     530011         Travis Head     1      3      0  ...   

       batter_rpm   batter_sr  batter_bpm bat_team_avg batter_avg_opp  \
14263   19.16

In [125]:
batting_cleaned=pd.read_csv('/content/batting_cleanedAllopp2.csv')
# Sort by batter, inns, and date
batting_cleaned.sort_values(by=['batter', 'inns', 'date'], inplace=True)

# Compute cumulative runs and dismissals before the match in each innings
batting_cleaned['cumulative_runs_in_innings'] = (
    batting_cleaned.groupby(['batter', 'inns'])['runs'].cumsum().shift(fill_value=0)
)
batting_cleaned['cumulative_dismissals_in_innings'] = (
    batting_cleaned.groupby(['batter', 'inns'])['isout'].cumsum().shift(fill_value=0)
)
batting_cleaned['cumulative_matches_in_innings'] = (
    batting_cleaned.groupby(['batter', 'inns'])['batter'].cumcount()
)
batting_cleaned['cumulative_balls_in_innings'] = (
    batting_cleaned.groupby(['batter', 'inns'])['balls'].cumsum().shift(fill_value=0)
)
batting_cleaned['cumulative_boundaries_in_innings'] = (
    batting_cleaned.groupby(['batter', 'inns'])['boundaries'].cumsum().shift(fill_value=0)
)

# Calculate batting average in innings before the match
batting_cleaned['batter_avg_inn'] = (
    batting_cleaned['cumulative_runs_in_innings'] / batting_cleaned['cumulative_dismissals_in_innings'].replace(0, pd.NA)
)
batting_cleaned['batter_rpm_inn'] = (
    batting_cleaned['cumulative_runs_in_innings'] / batting_cleaned['cumulative_matches_in_innings']
)
batting_cleaned['batter_bpm_inn'] = (
    batting_cleaned['cumulative_boundaries_in_innings'] / batting_cleaned['cumulative_matches_in_innings']
)
batting_cleaned['batter_sr_inn'] = (
    (batting_cleaned['cumulative_runs_in_innings'] * 100) / batting_cleaned['cumulative_balls_in_innings'].replace(0, pd.NA)
)

# Ensure first match in that innings has NaN
batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'inns']).head(1).index, 'batter_avg_inn'
] = pd.NA
batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'inns']).head(1).index, 'batter_rpm_inn'
] = pd.NA
batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'inns']).head(1).index, 'batter_bpm_inn'
] = pd.NA
batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'inns']).head(1).index, 'batter_sr_inn'
] = pd.NA

# Drop unnecessary columns
batting_cleaned.drop(columns=[
    'cumulative_dismissals_in_innings', 'cumulative_runs_in_innings',
    'cumulative_matches_in_innings', 'cumulative_balls_in_innings',
    'cumulative_boundaries_in_innings'
], inplace=True)

print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleaned_innings.csv', index=False)


            date  match_id        bat_team                    bowl_team  inns  \
2858  2017-04-19   1082611  Delhi Capitals          Sunrisers Hyderabad     2   
1999  2017-04-22   1082614  Delhi Capitals               Mumbai Indians     2   
2010  2017-05-06   1082635  Delhi Capitals               Mumbai Indians     2   
1604  2017-05-10   1082640  Delhi Capitals                Gujarat Lions     2   
2668  2017-05-14   1082646  Delhi Capitals  Royal Challengers Bangalore     2   

      batter_id       batter  runs  balls  fours  ...  batter_avg_opp  \
2858      30102  Zaheer Khan     0      0      0  ...             2.0   
1999      30102  Zaheer Khan     0      0      0  ...             2.0   
2010      30102  Zaheer Khan     2     11      0  ...             2.0   
1604      30102  Zaheer Khan     0      0      0  ...             4.0   
2668      30102  Zaheer Khan     1      1      0  ...             2.0   

      batter_rpm_opp  batter_bpm_opp batter_sr_opp bat_team_avg_opp  \
285

In [136]:
batting_cleaned=pd.read_csv('/content/batting_cleaned_innings.csv')
# Sort by bat_team, inns, and date
batting_cleaned.sort_values(by=['bat_team', 'inns', 'date'], inplace=True)

# Compute total runs scored by team per match in each innings
team_innings_runs = batting_cleaned.groupby(['match_id', 'bat_team', 'inns'])['runs'].sum().reset_index()

# Extract match dates directly from batting_cleaned
team_innings_runs = team_innings_runs.merge(
    batting_cleaned[['match_id', 'date']].drop_duplicates(),
    on='match_id',
    how='left'
)

# Sort by bat_team, inns, and date
team_innings_runs.sort_values(by=['bat_team', 'inns', 'date'], inplace=True)

# Compute cumulative runs in innings before the match
team_innings_runs['cumulative_team_runs_innings'] = (
    team_innings_runs.groupby(['bat_team', 'inns'])['runs'].cumsum().shift()
)

# Compute the number of past matches in that innings
team_innings_runs['cumulative_team_matches_innings'] = (
    team_innings_runs.groupby(['bat_team', 'inns']).cumcount()
)

# Calculate batting team average runs in innings before the match
team_innings_runs['bat_team_avg_inn'] = (
    team_innings_runs['cumulative_team_runs_innings'] / team_innings_runs['cumulative_team_matches_innings'].replace(0, pd.NA)
)

# Ensure first match in that innings is NaN
team_innings_runs.loc[
    team_innings_runs.groupby(['bat_team', 'inns']).head(1).index, 'bat_team_avg_inn'
] = pd.NA

# Merge back into batting_cleaned
batting_cleaned = batting_cleaned.merge(
    team_innings_runs[['match_id', 'bat_team', 'inns', 'bat_team_avg_inn']],
    on=['match_id', 'bat_team', 'inns'],
    how='left'
)
batting_cleaned.drop(columns=['bat_team_avg_inn_y'], errors='ignore', inplace=True)
batting_cleaned.rename(columns={'bat_team_avg_inn_x': 'bat_team_avg_opp'}, inplace=True)
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleaned_innings2.csv', index=False)


             date  match_id             bat_team     bowl_team  inns  \
14263  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14264  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14265  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14266  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14267  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   

       batter_id                  batter  runs  balls  fours  ...  \
14263    1132219            Sanvir Singh     6      4      1  ...   
14264    1159711           Shahbaz Ahmed     3      6      0  ...   
14265     802575             T Natarajan     0      0      0  ...   
14266     530011             Travis Head     0      1      0  ...   
14267    1153149  Vijayakanth Viyaskanth     0      0      0  ...   

       batter_rpm_opp  batter_bpm_opp  batter_sr_opp bat_team_avg_opp  \
14263        4.000000        1.000000     133.333333            153.0   
14264 

In [137]:
# Compute total runs scored by all teams per match in each innings
all_teams_innings_runs = batting_cleaned.groupby(['match_id', 'inns'])['runs'].sum().reset_index()

# Extract match dates directly from batting_cleaned
all_teams_innings_runs = all_teams_innings_runs.merge(
    batting_cleaned[['match_id', 'date']].drop_duplicates(),
    on='match_id',
    how='left'
)

# Sort by inns and date
all_teams_innings_runs.sort_values(by=['inns', 'date'], inplace=True)

# Compute cumulative runs in innings before the match
all_teams_innings_runs['cumulative_all_teams_runs_innings'] = (
    all_teams_innings_runs.groupby(['inns'])['runs'].cumsum().shift()
)

# Compute the number of past matches in that innings
all_teams_innings_runs['cumulative_all_teams_matches_innings'] = (
    all_teams_innings_runs.groupby(['inns']).cumcount()
)

# Calculate all teams' average runs in innings before the match
all_teams_innings_runs['all_team_avg_inn'] = (
    all_teams_innings_runs['cumulative_all_teams_runs_innings'] /
    all_teams_innings_runs['cumulative_all_teams_matches_innings'].replace(0, pd.NA)
)

# Ensure first match in that innings is NaN
all_teams_innings_runs.loc[
    all_teams_innings_runs.groupby(['inns']).head(1).index, 'all_team_avg_inn'
] = pd.NA

# Merge back into batting_cleaned
batting_cleaned = batting_cleaned.merge(
    all_teams_innings_runs[['match_id', 'inns', 'all_team_avg_inn']],
    on=['match_id', 'inns'],
    how='left'
)
batting_cleaned.drop(columns=['all_team_avg_inn_y'], errors='ignore', inplace=True)
batting_cleaned.rename(columns={'all_team_avg_inn_x': 'all_team_avg_inn'}, inplace=True)
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleaned_innings3.csv', index=False)


             date  match_id             bat_team     bowl_team  inns  \
14263  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14264  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14265  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14266  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   
14267  2024-05-19   1426307  Sunrisers Hyderabad  Punjab Kings     2   

       batter_id                  batter  runs  balls  fours  ...  \
14263    1132219            Sanvir Singh     6      4      1  ...   
14264    1159711           Shahbaz Ahmed     3      6      0  ...   
14265     802575             T Natarajan     0      0      0  ...   
14266     530011             Travis Head     0      1      0  ...   
14267    1153149  Vijayakanth Viyaskanth     0      0      0  ...   

       batter_bpm_opp  batter_sr_opp  bat_team_avg_opp all_team_avg_opp  \
14263        1.000000     133.333333             153.0       159.266187   
14

In [138]:
# Fill missing values in 'bat_team_avg' (Team's average runs per innings)
batting_cleaned['bat_team_avg'] = batting_cleaned.groupby(['bat_team', 'inns'])['bat_team_avg'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg'] = batting_cleaned.groupby(['bat_team', 'inns'])['bat_team_avg'].transform(lambda x: x.ffill())
batting_cleaned['bat_team_avg'] = batting_cleaned.groupby(['bat_team'])['bat_team_avg'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg'] = batting_cleaned.groupby(['bat_team'])['bat_team_avg'].transform(lambda x: x.ffill())
batting_cleaned['bat_team_avg'] = batting_cleaned.groupby(['inns'])['bat_team_avg'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg'] = batting_cleaned.groupby(['inns'])['bat_team_avg'].transform(lambda x: x.ffill())

# Fill missing values in 'all_team_avg_opp' (All teams' average runs per innings)
batting_cleaned['all_team_avg_opp'] = batting_cleaned.groupby(['inns'])['all_team_avg_opp'].transform(lambda x: x.bfill())
batting_cleaned['all_team_avg_opp'] = batting_cleaned.groupby(['inns'])['all_team_avg_opp'].transform(lambda x: x.ffill())


In [145]:
# Sort by batter, venue, and match date (chronologically)
batting_cleaned.sort_values(by=['batter', 'venue', 'date'], inplace=True)

# Compute cumulative runs and dismissals for each (batter, venue) group; shift to exclude current match.
batting_cleaned['cum_runs_venue'] = (
    batting_cleaned.groupby(['batter', 'venue'])['runs']
    .cumsum().shift()
)
batting_cleaned['cum_dismissals_venue'] = (
    batting_cleaned.groupby(['batter', 'venue'])['isout']
    .cumsum().shift()
)
batting_cleaned['cum_balls_venue'] = (
    batting_cleaned.groupby(['batter', 'venue'])['balls']
    .cumsum().shift()
)
batting_cleaned['cum_boundaries_venue'] = (
    batting_cleaned.groupby(['batter', 'venue'])['boundaries']
    .cumsum().shift()
)
batting_cleaned['cum_matches_venue'] = (
    batting_cleaned.groupby(['batter', 'venue']).cumcount()
)

# Calculate batsman average runs at the venue
batting_cleaned['batter_avg_venue'] = (
    batting_cleaned['cum_runs_venue'] /
    batting_cleaned['cum_dismissals_venue'].replace(0, pd.NA)
)
batting_cleaned['batter_sr_venue'] = (
    (batting_cleaned['cum_runs_venue'] * 100) /
    batting_cleaned['cum_balls_venue'].replace(0, pd.NA)
)
batting_cleaned['batter_rpm_venue'] = (
    batting_cleaned['cum_runs_venue'] /
    batting_cleaned['cum_matches_venue']
)
batting_cleaned['batter_bpm_venue'] = (
    batting_cleaned['cum_boundaries_venue'] /
    batting_cleaned['cum_matches_venue']
)

# Ensure that the very first match for each batter at that venue gets NaN.
batting_cleaned.loc[
    batting_cleaned.groupby(['batter', 'venue']).head(1).index,
    ['batter_avg_venue', 'batter_sr_venue', 'batter_rpm_venue', 'batter_bpm_venue']
] = pd.NA

# Clean up temporary columns.
batting_cleaned.drop(columns=['cum_runs_venue', 'cum_dismissals_venue', 'cum_matches_venue', 'cum_balls_venue', 'cum_boundaries_venue'], inplace=True)


print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleaned_Ven.csv', index=False)

            date  match_id        bat_team                    bowl_team  inns  \
2229  2015-05-12    829801  Delhi Capitals          Chennai Super Kings     2   
2317  2016-05-20    981003  Delhi Capitals          Sunrisers Hyderabad     2   
1524  2016-05-22    981011  Delhi Capitals  Royal Challengers Bangalore     1   
1447  2015-05-05    829783  Delhi Capitals               Mumbai Indians     1   
2350  2017-04-22   1082614  Delhi Capitals               Mumbai Indians     2   

      batter_id       batter  runs  balls  fours  ...  batter_avg_inn  \
2229      30102  Zaheer Khan     0      0      0  ...             NaN   
2317      30102  Zaheer Khan     0      0      0  ...             2.0   
1524      30102  Zaheer Khan     0      0      0  ...             4.0   
1447      30102  Zaheer Khan     0      0      0  ...             NaN   
2350      30102  Zaheer Khan     0      0      0  ...             3.0   

      batter_rpm_inn  batter_bpm_inn batter_sr_inn bat_team_avg_inn  \
222

In [150]:
# Create a per-match summary: total runs by the batting team at a given venue.
team_venue_runs = (
    batting_cleaned
    .groupby(['match_id', 'bat_team', 'venue'])['runs']
    .sum()
    .reset_index()
)

# Merge `date` from batting_cleaned to ensure chronological sorting.
team_venue_runs = team_venue_runs.merge(
    batting_cleaned[['match_id', 'date']].drop_duplicates(),
    on='match_id',
    how='left'
)

# Sort by bat_team, venue, and date.
team_venue_runs.sort_values(by=['bat_team', 'venue', 'date'], inplace=True)

# Compute cumulative team runs and count (number of past matches) within each (bat_team, venue) group.
team_venue_runs['cum_team_runs_venue'] = (
    team_venue_runs.groupby(['bat_team', 'venue'])['runs']
    .cumsum().shift()
)
team_venue_runs['cum_team_matches_venue'] = (
    team_venue_runs.groupby(['bat_team', 'venue']).cumcount()
)

# Compute batting team average runs on venue before the match.
team_venue_runs['bat_team_avg_venue'] = (
    team_venue_runs['cum_team_runs_venue'] /
    team_venue_runs['cum_team_matches_venue'].replace(0, pd.NA)
)

# Set the first match in each (bat_team, venue) group to NaN.
team_venue_runs.loc[
    team_venue_runs.groupby(['bat_team', 'venue']).head(1).index,
    'bat_team_avg_venue'
] = pd.NA

# Merge the team-level result back into batting_cleaned so that every player from the same match gets the same value.
batting_cleaned = batting_cleaned.merge(
    team_venue_runs[['match_id', 'bat_team', 'venue', 'bat_team_avg_venue']],
    on=['match_id', 'bat_team', 'venue'],
    how='left'
)
batting_cleaned.drop(columns=['bat_team_avg_venue_x'], errors='ignore', inplace=True)
batting_cleaned.rename(columns={'bat_team_avg_venue_y': 'bat_team_avg_venue'}, inplace=True)
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleaned_ven2.csv', index=False)


             date  match_id        bat_team                    bowl_team  \
14263  2015-05-12    829801  Delhi Capitals          Chennai Super Kings   
14264  2016-05-20    981003  Delhi Capitals          Sunrisers Hyderabad   
14265  2016-05-22    981011  Delhi Capitals  Royal Challengers Bangalore   
14266  2015-05-05    829783  Delhi Capitals               Mumbai Indians   
14267  2017-04-22   1082614  Delhi Capitals               Mumbai Indians   

       inns  batter_id       batter  runs  balls  fours  ...  batter_bpm_inn  \
14263     2      30102  Zaheer Khan     0      0      0  ...        0.000000   
14264     2      30102  Zaheer Khan     0      0      0  ...        0.000000   
14265     1      30102  Zaheer Khan     0      0      0  ...        0.166667   
14266     1      30102  Zaheer Khan     0      0      0  ...             NaN   
14267     2      30102  Zaheer Khan     0      0      0  ...        0.000000   

       batter_sr_inn  bat_team_avg_inn all_team_avg_inn batter

In [156]:
# Step 1: Compute total runs scored at each venue against a specific opposition in each match.
all_teams_venue_opp_runs = (
    batting_cleaned
    .groupby(['match_id', 'venue', 'bowl_team'])['runs']
    .sum()
    .reset_index()
)

# Step 2: Merge match dates from batting_cleaned.
all_teams_venue_opp_runs = all_teams_venue_opp_runs.merge(
    batting_cleaned[['match_id', 'date']].drop_duplicates(),
    on='match_id',
    how='left'
)

# Step 3: Sort by venue, opponent, and match date to maintain chronological order.
all_teams_venue_opp_runs.sort_values(by=['venue', 'bowl_team', 'date'], inplace=True)

# Step 4: Compute cumulative total runs at the venue against the opposition before the match.
all_teams_venue_opp_runs['cum_all_runs_venue_opp'] = (
    all_teams_venue_opp_runs.groupby(['venue', 'bowl_team'])['runs']
    .cumsum()
    .shift()
)

# Step 5: Compute cumulative number of matches played at the venue against the opposition.
all_teams_venue_opp_runs['cum_all_matches_venue_opp'] = (
    all_teams_venue_opp_runs.groupby(['venue', 'bowl_team']).cumcount()
)

# Step 6: Compute the sum of (runs/matches) for all teams at the venue against the opposition before the match.
all_teams_venue_opp_runs['all_team_avg_venue_opp'] = (
    all_teams_venue_opp_runs['cum_all_runs_venue_opp'] /
    all_teams_venue_opp_runs['cum_all_matches_venue_opp'].replace(0, pd.NA)
)

# Step 7: Ensure the first match at each venue against the opposition has NaN.
all_teams_venue_opp_runs.loc[
    all_teams_venue_opp_runs.groupby(['venue', 'bowl_team']).head(1).index,
    'all_team_avg_venue_opp'
] = pd.NA

# Step 8: Merge this opposition venue-based data back into batting_cleaned.
batting_cleaned = batting_cleaned.merge(
    all_teams_venue_opp_runs[['match_id', 'venue', 'bowl_team', 'all_team_avg_venue_opp']],
    on=['match_id', 'venue', 'bowl_team'],
    how='left'
)

batting_cleaned.drop(columns=['all_team_avg_venue_x'], errors='ignore', inplace=True)
batting_cleaned.rename(columns={'all_team_avg_venue_y': 'all_team_avg_venue'}, inplace=True)
print(batting_cleaned.tail())
batting_cleaned.to_csv('/content/batting_cleaned_ven3.csv', index=False)

             date  match_id        bat_team                    bowl_team  \
14263  2015-05-12    829801  Delhi Capitals          Chennai Super Kings   
14264  2016-05-20    981003  Delhi Capitals          Sunrisers Hyderabad   
14265  2016-05-22    981011  Delhi Capitals  Royal Challengers Bangalore   
14266  2015-05-05    829783  Delhi Capitals               Mumbai Indians   
14267  2017-04-22   1082614  Delhi Capitals               Mumbai Indians   

       inns  batter_id       batter  runs  balls  fours  ...  \
14263     2      30102  Zaheer Khan     0      0      0  ...   
14264     2      30102  Zaheer Khan     0      0      0  ...   
14265     1      30102  Zaheer Khan     0      0      0  ...   
14266     1      30102  Zaheer Khan     0      0      0  ...   
14267     2      30102  Zaheer Khan     0      0      0  ...   

       bat_team_avg_inn  all_team_avg_inn  batter_avg_venue batter_sr_venue  \
14263        146.166667        146.413043              <NA>            <NA>   


In [158]:
# Forward and backward fill for 'bat_team_avg_venue' (team's average runs at venue)
batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['bat_team', 'venue'])['bat_team_avg_venue'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['bat_team', 'venue'])['bat_team_avg_venue'].transform(lambda x: x.ffill())
batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['bat_team'])['bat_team_avg_venue'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['bat_team'])['bat_team_avg_venue'].transform(lambda x: x.ffill())
batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['venue'])['bat_team_avg_venue'].transform(lambda x: x.bfill())
batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['venue'])['bat_team_avg_venue'].transform(lambda x: x.ffill())

# Forward and backward fill for 'all_team_avg_venue' (all teams' average runs at venue)
batting_cleaned['all_team_avg_venue'] = batting_cleaned.groupby(['venue'])['all_team_avg_venue'].transform(lambda x: x.bfill())
batting_cleaned['all_team_avg_venue'] = batting_cleaned.groupby(['venue'])['all_team_avg_venue'].transform(lambda x: x.ffill())

# Fill remaining NaN values in 'all_team_avg_venue' with its median value
batting_cleaned['all_team_avg_venue'] = batting_cleaned['all_team_avg_venue'].fillna(batting_cleaned['all_team_avg_venue'].median())


<ipython-input-158-37ab7dc3208d>:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['bat_team', 'venue'])['bat_team_avg_venue'].transform(lambda x: x.bfill())
<ipython-input-158-37ab7dc3208d>:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  batting_cleaned['bat_team_avg_venue'] = batting_cleaned.groupby(['bat_team', 'venue'])['bat_team_avg_venue'].transform(lambda x: x.ffill())
<ipython-input-158-37ab7dc3208d>:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is de

In [159]:
import numpy as np
import pandas as pd

def add_rolling_averages(df):
    # Create a copy to avoid modifying the original DataFrame
    result_df = df.copy()

    # Sort by date and match_id to ensure correct temporal order
    result_df = result_df.sort_values(['date', 'match_id'])

    # Initialize new columns
    result_df['batter_rpm_last5'] = np.nan
    result_df['batter_bpm_last5'] = np.nan
    result_df['batter_avg_last5'] = np.nan
    result_df['batter_sr_last5'] = np.nan

    # Group by batter to process each player separately
    for batter in result_df['batter'].unique():
        player_df = result_df[result_df['batter'] == batter].copy()

        # Calculate match-level aggregates for per-match metrics
        match_level = player_df.groupby('match_id').agg({
            'runs': 'sum',
            'boundaries': 'sum'
        }).reset_index()

        # Calculate rolling averages per match (including DNB matches)
        match_level['rolling_runs'] = match_level['runs'].shift().rolling(
            window=5, min_periods=1).mean()
        match_level['rolling_boundaries'] = match_level['boundaries'].shift().rolling(
            window=5, min_periods=1).mean()

        # Map these back to the main DataFrame
        match_rolling_stats = match_level.set_index('match_id')
        result_df.loc[result_df['batter'] == batter, 'batter_rpm_last5'] = \
            result_df[result_df['batter'] == batter]['match_id'].map(match_rolling_stats['rolling_runs'])
        result_df.loc[result_df['batter'] == batter, 'batter_bpm_last5'] = \
            result_df[result_df['batter'] == batter]['match_id'].map(match_rolling_stats['rolling_boundaries'])

        # For innings-based calculations, filter out DNB innings
        batting_innings = player_df[player_df['wickettype'] != 'DNB'].copy()

        # Calculate cumulative dismissals for average calculation
        batting_innings['is_dismissal'] = batting_innings['isout']

        # Create windows for innings-based calculations
        for idx in batting_innings.index:
            current_date = batting_innings.loc[idx, 'date']

            # Get previous 5 innings where player batted
            prev_innings = batting_innings[
                (batting_innings['date'] < current_date) |
                ((batting_innings['date'] == current_date) &
                 (batting_innings.index < idx))
            ].tail(5)

            if not prev_innings.empty:
                # Calculate runs per dismissal
                total_runs = prev_innings['runs'].sum()
                total_dismissals = prev_innings['is_dismissal'].sum()
                avg_runs = total_runs / max(total_dismissals, 1)  # Avoid division by zero

                # Calculate strike rate
                total_balls = prev_innings['balls'].sum()
                strike_rate = (total_runs / total_balls * 100) if total_balls > 0 else np.nan

                result_df.loc[idx, 'batter_avg_last5'] = avg_runs
                result_df.loc[idx, 'batter_sr_last5'] = strike_rate

    return result_df


In [160]:
batting_cleaned = add_rolling_averages(batting_cleaned)

In [161]:
print(batting_cleaned.tail())
#batting_cleaned.to_csv('/content/batting_cleaned_last5.csv', index=False

             date  match_id               bat_team              bowl_team  \
12706  2024-05-26   1426312    Sunrisers Hyderabad  Kolkata Knight Riders   
12911  2024-05-26   1426312    Sunrisers Hyderabad  Kolkata Knight Riders   
13237  2024-05-26   1426312  Kolkata Knight Riders    Sunrisers Hyderabad   
13310  2024-05-26   1426312  Kolkata Knight Riders    Sunrisers Hyderabad   
13373  2024-05-26   1426312  Kolkata Knight Riders    Sunrisers Hyderabad   

       inns  batter_id              batter  runs  balls  fours  ...  \
12706     1     802575         T Natarajan     0      0      0  ...   
12911     1     530011         Travis Head     0      1      0  ...   
13237     2    1209292       Vaibhav Arora     0      0      0  ...   
13310     2    1108375  Varun Chakravarthy     0      0      0  ...   
13373     2     851403      Venkatesh Iyer    52     26      4  ...   

       batter_rpm_venue  batter_bpm_venue bat_team_avg_venue  \
12706               0.0               0.0     

In [162]:
# Columns to apply imputation
cols = ['batter_avg_last5', 'batter_sr_last5', 'batter_rpm_last5', 'batter_bpm_last5']

# Apply imputation efficiently
for col in cols:
    # Perform both backward and forward fill using transform (keeps original index)
    batting_cleaned[col] = (
        batting_cleaned.groupby(['batter'])[col]
        .transform(lambda x: x.bfill().ffill())
    )

    # Compute median once
    median = batting_cleaned[col].median()

    # Fill remaining NaNs with median/2
    batting_cleaned[col].fillna(median / 2, inplace=True)


<ipython-input-162-e6e27d7ff809>:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  batting_cleaned[col].fillna(median / 2, inplace=True)
<ipython-input-162-e6e27d7ff809>:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)',

In [164]:
batting_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14268 entries, 105 to 13373
Data columns (total 47 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    14268 non-null  object 
 1   match_id                14268 non-null  int64  
 2   bat_team                14268 non-null  object 
 3   bowl_team               14268 non-null  object 
 4   inns                    14268 non-null  int64  
 5   batter_id               14268 non-null  int64  
 6   batter                  14268 non-null  object 
 7   runs                    14268 non-null  int64  
 8   balls                   14268 non-null  int64  
 9   fours                   14268 non-null  int64  
 10  sixes                   14268 non-null  int64  
 11  sr                      14268 non-null  float64
 12  isout                   14268 non-null  int64  
 13  wickettype              14268 non-null  object 
 14  bowler_id               7285 non-null   o

In [165]:
batting_cleaned.to_csv('/content/batting_Features.csv', index=False)